# FPR Experiment for Data-Split p-values

This notebook estimates the null false positive rate (FPR) of a data-split baseline for Adaptive CoRT.

For each run, the target dataset is split into two halves, while the source datasets are kept intact:

- the selection half is used to run adaptive source selection and the final CoRT fit,
- the inference half is used to construct $\eta$ and compute an ordinary two-sided Gaussian p-value.

This mirrors the naive p-value baseline, except the target model-selection step and the target p-value calculation are separated onto disjoint target samples.

In [6]:
# # For kaggle:
# !git clone https://github.com/maTiahK/CORT_SI.git
# %cd /kaggle/working/CORT_SI

# !python -m pip install -q --upgrade pip
# !python -m pip install -q -e .

# from cort_si import gen_data
# print("CORT_SI import OK")

In [7]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO_ROOT = next((candidate for candidate in candidates if (candidate / "cort_si").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root containing the cort_si package.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cort_si import (
    adaptive_source_selection,
    construct_Y,
    construct_Sigma,
    construct_folds,
    construct_test_statistic,
    gen_data,
    solve_cort_model,
)

RESULTS_DIR = REPO_ROOT / "run_experiment" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
p = 300
nS = 100
nT = 50
num_info_aux = 3
num_uninfo_aux = 2
K = num_info_aux + num_uninfo_aux
lambda_sel = 0.25
lambda0 = 0.5
T = 3

num_runs = 100
seed = 0

alpha = 0.05
true_beta_scale = 0.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False

print(f"num_runs = {num_runs}")
print(f"fixed alpha = {alpha}")
print(f"Null model true_beta scale = {true_beta_scale}")
print(f"K = {K}, p = {p}, nS = {nS}, nT = {nT}")
print("The target is split into a selection half and an inference half at every run; sources are kept intact.")

num_runs = 1500
fixed alpha = 0.05
Null model true_beta scale = 0.0
K = 5, p = 300, nS = 100, nT = 50
The target is split into a selection half and an inference half at every run; sources are kept intact.


In [9]:
def generate_null_instance(iteration_seed):
    return gen_data.generate_data(
        p=p,
        nS=nS,
        nT=nT,
        K=K,
        true_beta=true_beta_scale,
        rho=rho,
        sigma_noise=sigma_noise,
        source_shift_sd=source_shift_sd,
        covariate_shift=covariate_shift,
        seed=iteration_seed,
        num_info_aux=num_info_aux,
        num_uninfo_aux=num_uninfo_aux
        
    )


def split_indices_in_half(num_rows, rng):
    shuffled = rng.permutation(num_rows)
    selection_idx, inference_idx = [np.sort(block.astype(int)) for block in np.array_split(shuffled, 2)]
    if selection_idx.size == 0 or inference_idx.size == 0:
        return None, None
    return selection_idx, inference_idx


def split_dataset_block(X, Y, Sigma, rng):
    selection_idx, inference_idx = split_indices_in_half(X.shape[0], rng)
    if selection_idx is None:
        return None

    Sigma = np.asarray(Sigma)
    return {
        "X_selection": X[selection_idx],
        "Y_selection": np.asarray(Y)[selection_idx],
        "Sigma_selection": Sigma[np.ix_(selection_idx, selection_idx)],
        "X_inference": X[inference_idx],
        "Y_inference": np.asarray(Y)[inference_idx],
        "Sigma_inference": Sigma[np.ix_(inference_idx, inference_idx)],
    }


def split_instance_for_datasplit(XS_list, YS_list, SigmaS_list, X0, Y0, Sigma0, split_seed):
    rng = np.random.default_rng(split_seed)

    target_split = split_dataset_block(X0, Y0, Sigma0, rng)
    if target_split is None:
        return None

    return {
        "XS_selection_list": XS_list,
        "YS_selection_list": YS_list,
        "SigmaS_selection_list": SigmaS_list,
        "X0_selection": target_split["X_selection"],
        "Y0_selection": target_split["Y_selection"],
        "Sigma0_selection": target_split["Sigma_selection"],
        "XS_inference_list": XS_list,
        "YS_inference_list": YS_list,
        "SigmaS_inference_list": SigmaS_list,
        "X0_inference": target_split["X_inference"],
        "Y0_inference": target_split["Y_inference"],
        "Sigma0_inference": target_split["Sigma_inference"],
    }


def compute_random_selected_datasplit_pvalue(X0, Y0, XS_list, YS_list, SigmaS_list, Sigma0, lambdak_list, split_seed, feature_seed):
    split_data = split_instance_for_datasplit(XS_list, YS_list, SigmaS_list, X0, Y0, Sigma0, split_seed)
    if split_data is None:
        return None

    folds = construct_folds(split_data["X0_selection"].shape[0], T=T, shuffle=False)
    I_obs = adaptive_source_selection(
        split_data["X0_selection"],
        split_data["Y0_selection"],
        split_data["XS_selection_list"],
        split_data["YS_selection_list"],
        folds,
        lambda_sel,
        verbose=False,
    )
    _, beta0_hat, _, _ = solve_cort_model(
        split_data["X0_selection"],
        split_data["Y0_selection"],
        split_data["XS_selection_list"],
        split_data["YS_selection_list"],
        I_obs,
        lambda0,
        lambdak_list,
        verbose=False,
        label="Observed CoRT fit on selection half",
    )
    M_obs = [idx for idx, value in enumerate(beta0_hat) if value != 0]
    if not M_obs:
        return None

    feature_idx = random.Random(feature_seed).choice(M_obs)
    Y_inference = construct_Y(split_data["YS_inference_list"], split_data["Y0_inference"])
    Sigma_inference = construct_Sigma(split_data["SigmaS_inference_list"], split_data["Sigma0_inference"])
    X0M_inference = split_data["X0_inference"][:, M_obs]
    etaj, etajTy = construct_test_statistic(
        feature_idx,
        X0M_inference,
        Y_inference,
        M_obs,
        split_data["Y0_inference"].shape[0],
        Y_inference.shape[0],
    )

    variance = float(np.asarray(etaj.T @ Sigma_inference @ etaj).item())
    if not np.isfinite(variance) or variance <= 0.0:
        return None

    z_score = float(etajTy / np.sqrt(variance))
    datasplit_pvalue = float(2.0 * stats.norm.sf(abs(z_score)))

    return {
        "feature_idx": int(feature_idx),
        "datasplit_pvalue": datasplit_pvalue,
        "selected_features": int(len(M_obs)),
        "selected_sources": int(len(I_obs)),
        "selection_target_size": int(split_data["Y0_selection"].shape[0]),
        "inference_target_size": int(split_data["Y0_inference"].shape[0]),
    }


def summarize_fpr_at_alpha(pvalues, alpha):
    if pvalues.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    rejection_indicators = (pvalues <= alpha).astype(float)
    running_fpr = np.cumsum(rejection_indicators) / np.arange(1, rejection_indicators.size + 1)
    return rejection_indicators, running_fpr


def run_fpr_experiment(num_runs, seed):
    datasplit_pvalues = []
    feature_indices = []
    selected_feature_counts = []
    selected_source_counts = []
    selection_target_sizes = []
    inference_target_sizes = []

    for iteration in range(num_runs):
        data_seed = seed + iteration
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(data_seed)
        lambdak_list = [lambda0] * len(XS_list)
        pvalue_record = compute_random_selected_datasplit_pvalue(
            X0,
            Y0,
            XS_list,
            YS_list,
            SigmaS_list,
            Sigma0,
            lambdak_list,
            split_seed=data_seed + 10000,
            feature_seed=data_seed + 20000,
        )

        if pvalue_record is not None:
            datasplit_pvalues.append(pvalue_record["datasplit_pvalue"])
            feature_indices.append(pvalue_record["feature_idx"])
            selected_feature_counts.append(pvalue_record["selected_features"])
            selected_source_counts.append(pvalue_record["selected_sources"])
            selection_target_sizes.append(pvalue_record["selection_target_size"])
            inference_target_sizes.append(pvalue_record["inference_target_size"])

        if (iteration + 1) % 10 == 0:
            print(f"Completed {iteration + 1}/{num_runs} runs")

    return {
        "datasplit_pvalues": np.asarray(datasplit_pvalues, dtype=float),
        "feature_indices": np.asarray(feature_indices, dtype=int),
        "selected_feature_counts": np.asarray(selected_feature_counts, dtype=int),
        "selected_source_counts": np.asarray(selected_source_counts, dtype=int),
        "selection_target_sizes": np.asarray(selection_target_sizes, dtype=int),
        "inference_target_sizes": np.asarray(inference_target_sizes, dtype=int),
    }

In [ ]:
experiment_results = run_fpr_experiment(num_runs=num_runs, seed=seed)
datasplit_rejections, datasplit_running_fpr = summarize_fpr_at_alpha(experiment_results["datasplit_pvalues"], alpha)

fpr_path = RESULTS_DIR / "fpr_datasplit_results.npz"
np.savez(
    fpr_path,
    datasplit_pvalues=experiment_results["datasplit_pvalues"],
    feature_indices=experiment_results["feature_indices"],
    selected_feature_counts=experiment_results["selected_feature_counts"],
    selected_source_counts=experiment_results["selected_source_counts"],
    selection_target_sizes=experiment_results["selection_target_sizes"],
    inference_target_sizes=experiment_results["inference_target_sizes"],
    datasplit_rejections=datasplit_rejections,
    datasplit_running_fpr=datasplit_running_fpr,
    alpha=np.asarray([alpha], dtype=float),
    num_runs=np.asarray([num_runs], dtype=int),
    p=np.asarray([p], dtype=int),
)

print(f"Saved data-split FPR results to {fpr_path}")
print(f"Valid null draws collected: {experiment_results['datasplit_pvalues'].size}/{num_runs}")
if datasplit_rejections.size > 0:
    print(f"Empirical data-split FPR at alpha={alpha:.2f}: {datasplit_rejections.mean():.3f}")
if experiment_results['selected_feature_counts'].size > 0:
    print(f"Average selected features from the selection half: {experiment_results['selected_feature_counts'].mean():.2f}")
    print(f"Average selected sources from the selection half: {experiment_results['selected_source_counts'].mean():.2f}")

Completed 10/1500 runs
Completed 20/1500 runs
Completed 30/1500 runs


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if datasplit_running_fpr.size > 0:
    draw_index = np.arange(1, datasplit_running_fpr.size + 1)
    axes[0].plot(draw_index, datasplit_running_fpr, color="#1f5673", linewidth=2, label="Data-split running FPR")
    axes[0].axhline(alpha, color="#35524a", linestyle="--", linewidth=2, label=f"Reference alpha = {alpha:.2f}")
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "No valid null p-values", ha="center", va="center", transform=axes[0].transAxes)

axes[0].set_title(f"Data-split running FPR at alpha={alpha:.2f}")
axes[0].set_xlabel("Number of valid null draws")
axes[0].set_ylabel("Cumulative rejection rate")
axes[0].set_xlim(1, max(1, datasplit_running_fpr.size))
axes[0].set_ylim(0.0, 1.0)
axes[0].grid(alpha=0.25)

axes[1].hist(experiment_results["datasplit_pvalues"], bins=np.linspace(0.0, 1.0, 11), density=True, color="#9fc490", edgecolor="#35524a", alpha=0.85)
axes[1].axhline(1.0, color="#7a3b2e", linestyle="--", linewidth=2, label="Uniform density = 1")
axes[1].set_title("Data-split null p-values")
axes[1].set_xlabel("Data-split p-value")
axes[1].set_ylabel("Density")
axes[1].set_xlim(0.0, 1.0)
axes[1].grid(alpha=0.25)
axes[1].legend()

feature_bins = np.arange(0, max(1, int(experiment_results["selected_feature_counts"].max()) if experiment_results["selected_feature_counts"].size > 0 else 1) + 2) - 0.5
axes[2].hist(experiment_results["selected_feature_counts"], bins=feature_bins, color="#d8896b", edgecolor="#7a3b2e", alpha=0.85)
axes[2].set_title("Selected target features on selection half")
axes[2].set_xlabel("Number of selected target features")
axes[2].set_ylabel("Count")
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()